# braid example: ERA5 climatology via region writes

Compute monthly climatology (mean 2m temperature by calendar month) across
multiple years of ERA5 hourly data. Each Lambda worker handles one year,
groups by calendar month using flox, and writes its result directly to a
shared S3 zarr store — no large arrays are returned through braid.

**Dataset**: [ARCO ERA5](https://github.com/google-research/arco-era5) on GCS,
`gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3`  
**Chunks**: `(1, 721, 1440)` — one hour per chunk, full global grid, ~4 MB each  
**Pattern**: one Lambda per year → `groupby(time.month).mean()` → region write  
**Dependencies**: `xarray`, `zarr`, `obstore`, `flox`

```bash
uv add braid xarray zarr obstore flox
```

## Why one year per worker?

ERA5 chunks are `(1, 721, 1440)` — each chunk is one hour of the full global
grid. Spatial sub-tiling would read the same chunks anyway. Temporal chunking
is natural: one year = ~8760 chunks = ~34 GB of S3 reads, but flox streams
the groupby reduction so **peak Lambda memory is ~230 MB** — it holds only
accumulators (12 groups × full grid ≈ 100 MB) plus a small in-flight window,
never the full year in memory at once.

Output per worker: `(12, 721, 1440)` ≈ 50 MB, written directly to the output
zarr store. braid collects only a small status dict.

In [ ]:
# /// script
# requires-python = ">=3.13"
# dependencies = [
#   "braid",
#   "xarray",
#   "zarr",
#   "obstore",
#   "flox",
#   "dask",
#   "matplotlib",
# ]
# ///

## 1. Configuration

In [ ]:
import logging
import os
logging.basicConfig(level=logging.INFO, format="%(message)s")

# Prevent obstore from polling EC2 instance metadata (169.254.169.254) for credentials.
# Not available locally; Lambda credentials come from env vars, not IMDS.
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

SOURCE = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
VARIABLE = "2m_temperature"
OUTPUT_STORE = "s3://carbonplan-share/era5-monthly-clim.zarr"  # replace

# Full ERA5 record: 1940–2024, 85 years, 85 Lambda workers.
# Cost: GCS reads free (public bucket), Lambda compute ~$0.005/worker = ~$0.43 total.
START_YEAR = 1940
END_YEAR = 2024

## 2. Initialize braid

In [ ]:
import braid

# config = braid.init(
#     bucket="my-bucket",
#     region="us-west-2",
#     create_role=True,
#     memory_mb=1024,
#     timeout_s=600,
# )
# arn = braid.syncarn(extra_dep_groups=["worker"])
# print(f"Layer: {}")

## 3. Inspect the source dataset

In [ ]:
import zarr
import xarray as xr
from obstore.store import from_url
from zarr.storage import ObjectStore

store = from_url(SOURCE, skip_signature=True)
with zarr.config.set({"async.concurrency": 32}):
    ds = xr.open_dataset(ObjectStore(store), engine="zarr", chunks={})[[VARIABLE]]

print(ds)
print("\nchunks:", ds[VARIABLE].chunks[:3], "...")

lat = ds.latitude.values
lon = ds.longitude.values
print(f"Grid: {len(lat)} lat × {len(lon)} lon")

## 4. Build the template output store

Shape `(n_years, 12, 721, 1440)` — one entry per year × month.
`compute=False` writes only zarr metadata; workers fill the data.

In [ ]:
import numpy as np
from obstore.store import from_url
from zarr.storage import ObjectStore
import boto3

years = np.arange(START_YEAR, END_YEAR + 1)
months = np.arange(1, 13)
n_years = len(years)

template = xr.Dataset(
    {
        VARIABLE: xr.DataArray(
            data=np.full((n_years, 12, len(lat), len(lon)), np.nan, dtype=np.float32),
            dims=["year", "month", "latitude", "longitude"],
            coords={
                "year": years,
                "month": months,
                "latitude": lat,
                "longitude": lon,
            },
            attrs=ds[VARIABLE].attrs,
        )
    },
    attrs={"source": SOURCE, "created_with": "braid"},
)


session = boto3.Session()
creds = session.get_credentials().get_frozen_credentials()              

output_zstore = ObjectStore(from_url(                                   
    OUTPUT_STORE,
    region="us-west-2",                                                 
    aws_access_key_id=creds.access_key,
    aws_secret_access_key=creds.secret_key,                             
))           

template.to_zarr(
    output_zstore,
    mode="w",
    compute=False,
    encoding={
        VARIABLE: {
            "chunks": (1, 12, 721, 1440),
            "dtype": "float32",
        }
    },
)
print(f"Template written: {OUTPUT_STORE}")
print(template)

In [ ]:
OUTPUT_STORE

## 5. Define the worker function

Each Lambda:
1. Opens ERA5 on GCS, selects one year lazily
2. Runs `groupby("time.month").mean()` via flox —
3. Writes `(1, 12, 721, 1440)` to the correct year index in the output store
4. Returns a small status dict — arrays never cross the Lambda return path

In [ ]:
def compute_year_climatology(year_idx: tuple[int, int]) -> dict:
    """
    year_idx: (year, index_into_output_year_dim)
    """
    import zarr
    import xarray as xr
    from obstore.store import from_url
    from zarr.storage import ObjectStore

    year, idx = year_idx

    source = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
    output = OUTPUT_STORE  # captured via cloudpickle
    variable = VARIABLE

    with zarr.config.set({"async.concurrency": 32}):
        # Read: GCS via obstore, chunked with dask for lazy groupby
        src_store = ObjectStore(from_url(source, skip_signature=True))
        ds = xr.open_dataset(src_store, engine="zarr", chunks={})[[variable]]
        year_data = ds[variable].sel(time=str(year))

        # flox streams the groupby — accumulates sum+count per group,
        # never materialises the full year in memory
        monthly = (
            year_data
            .groupby("time.month")
            .mean(engine="flox")
            .compute()
        )

        # Write: S3 via obstore
        monthly = monthly.expand_dims("year").assign_coords(year=[year])
        out_store = ObjectStore(from_url(output))
        monthly.to_dataset(name=variable).to_zarr(
            out_store,
            region={"year": slice(idx, idx + 1), "month": slice(None)},
            consolidated=False
        )

    return {
        "year": year,
        "idx": idx,
        "global_mean_k": float(monthly.mean()),
        "status": "ok",
    }

## 6. Dry run

In [ ]:
from braid import fan

inputs = [(int(y), i) for i, y in enumerate(years)]
print(f"{len(inputs)} workers: {inputs}")

fan(compute_year_climatology, inputs[:1], dry_run=True)

## 7. Dispatch

Each worker reads ~34 GB from GCS and writes ~50 MB to S3.
braid collects only the small status dicts.

In [ ]:
import time

t0 = time.time()
statuses = fan(
    compute_year_climatology,
    inputs,
    batch_size=1,
    timeout_s=600,
)
elapsed = time.time() - t0

failed = [s for s in statuses if s.get("status") != "ok"]
print(f"{len(statuses)} years in {elapsed:.1f}s — {len(failed)} failed")

## 8. Verify and plot

In [ ]:
from obstore.store import from_url
from zarr.storage import ObjectStore

out = xr.open_dataset(ObjectStore(from_url(OUTPUT_STORE)), engine="zarr", chunks=None)
print(out)

n_nan = int(out[VARIABLE].isnull().sum())
print(f"NaN: {n_nan} / {out[VARIABLE].size} — {'complete' if n_nan == 0 else 'INCOMPLETE'}")

In [ ]:
import matplotlib.pyplot as plt

# July mean across all years — load one slice, no dask needed
july_mean = (out[VARIABLE].sel(month=7).mean("year").values - 273.15)

fig, ax = plt.subplots(figsize=(12, 5))
img = ax.pcolormesh(
    out.longitude.values, out.latitude.values, july_mean,
    cmap="RdBu_r", vmin=-40, vmax=40,
)
plt.colorbar(img, ax=ax, label="°C")
ax.set_title(f"ERA5 July mean 2m temperature — {START_YEAR}–{END_YEAR}")
plt.tight_layout()

## Notes

**Memory budget**: flox groupby streams the reduction — accumulators are
`n_months × 721 × 1440 × float32 × 2` ≈ 100 MB. Set `async.concurrency`
lower (e.g. 16) to reduce peak if hitting memory limits.

**Failure recovery**: workers are idempotent — re-run with only the failed
`(year, idx)` pairs. Their output slice is simply overwritten.

**Returning arrays vs region writes**: at 50 MB per worker, braid would
route results through S3 automatically. Region writes avoid the
collect-then-reassemble step and keep the output store as the single source
of truth.

**Multi-stage pipelines**: computing anomalies (value − climatology) requires
this climatology as input to a second round of workers. See
`docs/cubed_executor.md` for automating multi-stage plans via Cubed.